# Graded Activity: Let's revisit the flow problem as a linear algebra system
In this activity, we'll reformulate the flow problem as a linear algebra system and solve it using our iterative solvers and the QR iteration method.

__Learning objectives:__ Fill me in.

Let's go!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's setup our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl"));

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material. 

### Constants
Let's define some constants that will be used throughout the notebook. See the comment for a description of each constant, what it represents, its value, units, etc.

In [2]:
# We are going to plot the path through a graph, so let's provide the coordinates for each node, i.e., the layout
# This layout looks like our schematic but you can rearrange this if you want!
node_coordinates = [

    # warehouse node (you)
    10.0 10.0 ; # 1 warehouse node s (x,y) coordinates

    # processing nodes (workers)
    11.0 11.0 ; # 2 incoming shipping processing node (x,y) coordinates
    11.0 10.0 ; # 3 incoming shipping processing node (x,y) coordinates
    11.0 9.0 ; # 4 incoming shipping processing node (x,y) coordinates

    # job nodes (tasks)
    12.0 11.0 ; # 5 job node (x,y) coordinates
    12.0 10.0 ; # 6 job node (x,y) coordinates
    12.0 9.0 ; # 7 job node (x,y) coordinates
    12.0 8.0 ; # 8 job node (x,y) coordinates

    # sink nodes (targets, tasks done!)
    13.0 11.0 ; # 9 sink node t (x,y) coordinates
    13.0 10.0 ; # 10 sink node t (x,y) coordinates
    13.0 9.0 ; # 11 sink node t (x,y) coordinates
    13.0 8.0 ; # 12 sink node t (x,y) coordinates
    
    14.0 10.0 ; # 13 end node (x,y) coordinates
];

### Helper methods
In this activity, we'll include a helper method for computing the linearly independent columns of a matrix. Later, we'll use this method to identify the flows that we can measure. This method use a QR decomposition to find the pivot columns of the matrix, which correspond to the linearly independent columns.

In [50]:
# Return indices of independent columns and numerical rank
function select_independent_columns(A; k::Union{Int,Nothing}=nothing, tol::Union{Float64,Nothing}=nothing)
    n, m = size(A)
    F = qr(A, Val(true))               # column-pivoted QR: A*Perm = Q*R
    p = F.p
    R = F.R
    # Numerical rank via diagonal of R with a standard tolerance
    if tol === nothing
        tol = max(n, m) * eps(real(eltype(A))) * abs(R[1,1])
    end
    d = abs.(diag(R))
    r = findlast(>(tol), d)
    r = r === nothing ? 0 : r
    kk = k === nothing ? min(n, r) : min(k, r)
    keep = p[1:kk]
    drop = setdiff(1:m, keep)
    return keep, drop, r, tol
end;

___

## Task 1: Build a production process graph model
In this task, we'll build a graph model for our production process. We'll use this model to compute a constraint matrix $\mathbf{A}$.

The problem graph edges are stored in `data/Production-Process-Bipartite.edgelist` with fields: 

> __Records__: Each record in our edgelist file has the comma separated fields: `source,` `target,` `cost,` `lb capacity,` `ub capacity`. The `source` field is the id for the source node, e.g., `1`, the `target` field is the target node id, the `cost` is the cost of assigning the source node to the target node, the `lb capacity` is the lower bound capacity for the edge, and the `ub capacity` is the upper bound capacity for the edge.

In this activity, we'll ignore the edge weight (set to `1`), and instead will focus on the capacity constraints. Ok, so now let's setup our edge parser __callback function__:

In [3]:
"""
    function edgerecordparser(record::String, delim::Char=',') -> Tuple{Int, Int, Float64} | Nothing

This method is called to parse a single edge record from the edgelist file. It gets called once for each record in the file. 
The function splits the record into fields based on the specified delimiter and extracts the source node, target node, and cost (weight) of the edge. 
It returns a tuple containing these values. If the record does not have the expected number of fields, it returns `nothing`.

### Arguments
- `record`: The edge record string to parse.
- `delim`: The delimiter used to split the record.

### Returns
- A tuple containing the source node, target node, and cost of the edge, or `nothing` if the record is invalid.
"""
function edgerecordparser(record::String, delim::Char=',')
    
    # record (five fields)
    # source, target, cost, lb, ub

    fields = split(record, delim) # this assumes a record of the form "source,target,weight"
    if length(fields) < 5 # we have 5 fields
        return nothing
    end

    # get my data from the line -
    source = parse(Int, fields[1]) # source id
    target = parse(Int, fields[2]) # target id
    cost = parse(Float64, fields[3]) # edge weight
    l = parse(Float64, fields[4]) # lower bound capacity
    u = parse(Float64, fields[5]) # upper bound capacity

    # return a tuple -
    return (source, target, cost, l, u)
end;

Next, let's set the path to the edge list file in the `path_to_edge_file::String` variable:

In [4]:
path_to_edge_file = joinpath(_PATH_TO_DATA, "Production-Process-Bipartite.edgelist"); # this points to the graph shown above

Next, construct a dictionary [of `MyConstrainedGraphEdgeModel` instances](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MyConstrainedGraphEdgeModel) which stores the data for the edges. Let's save our edge models in the `myedgemodels::Dict{Int64, MyConstrainedGraphEdgeModel}` dictionary.

The keys in the edge dictionary will be the edge ids (which we can assume are unique), and the values will be the corresponding `MyConstrainedGraphEdgeModel` instances. Here, we've used the line index in the edgefile as the edge id.

In [5]:
myedgemodels = MyConstrainedGraphEdgeModels(path_to_edge_file, edgerecordparser, delim=',', comment='#')

Dict{Int64, MyConstrainedGraphEdgeModel} with 23 entries:
  5  => MyConstrainedGraphEdgeModel(5, 7, 11, 1.0, 0.0, 1.0)
  16 => MyConstrainedGraphEdgeModel(16, 3, 6, 1.0, 0.0, 1.0)
  20 => MyConstrainedGraphEdgeModel(20, 4, 6, 1.0, 0.0, 1.0)
  12 => MyConstrainedGraphEdgeModel(12, 2, 6, 1.0, 0.0, 1.0)
  8  => MyConstrainedGraphEdgeModel(8, 10, 13, 1.0, 0.0, 1.0)
  17 => MyConstrainedGraphEdgeModel(17, 3, 7, 1.0, 0.0, 1.0)
  1  => MyConstrainedGraphEdgeModel(1, 1, 3, 1.0, 0.0, 1.0)
  19 => MyConstrainedGraphEdgeModel(19, 4, 5, 1.0, 0.0, 1.0)
  0  => MyConstrainedGraphEdgeModel(0, 1, 2, 1.0, 0.0, 1.0)
  22 => MyConstrainedGraphEdgeModel(22, 4, 8, 1.0, 0.0, 1.0)
  6  => MyConstrainedGraphEdgeModel(6, 8, 12, 1.0, 0.0, 1.0)
  11 => MyConstrainedGraphEdgeModel(11, 2, 5, 1.0, 0.0, 1.0)
  9  => MyConstrainedGraphEdgeModel(9, 11, 13, 1.0, 0.0, 1.0)
  14 => MyConstrainedGraphEdgeModel(14, 2, 8, 1.0, 0.0, 1.0)
  3  => MyConstrainedGraphEdgeModel(3, 5, 9, 1.0, 0.0, 1.0)
  7  => MyConstrainedGraphEd

Finally, we can build a graph instance. Since this is a directed graph, we'll construct [a `MyDirectedBipartiteGraphModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MyDirectedBipartiteGraphModel) using [a `build(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.build). Let's save our graph model in the `directedgraphmodel::MyDirectedBipartiteGraphModel` variable.

In [6]:
directedgraphmodel = let

    # initialize -
    s = 1; # what is the source node
    t = 13; # what is the sink node

    # call the build method to create the graph model
    model = build(MyDirectedBipartiteGraphModel, (
        s = s, # source index
        t = t, # sink index
        edges = myedgemodels
    ));

    
    model # return the model 
end;

In [7]:
directedgraphmodel |> typeof |> T-> fieldnames(T)

(:nodes, :edges, :children, :edgesinverse, :left, :right, :source, :sink, :capacity)

Let's build a map between the $(u,v)$ pairs and their corresponding edge indices. We'll call this the `edge_index_map::Dict{Tuple{Int,Int},Int}`.


In [8]:
edge_index_map = let

    # initialize -
    edge_index_map = Dict{Tuple{Int,Int},Int}() # (u,v) -> edge index
    for (k, (u,v)) ∈ directedgraphmodel.edgesinverse
        edge_index_map[(u,v)] = k
    end
    edge_index_map # return
end

Dict{Tuple{Int64, Int64}, Int64} with 23 entries:
  (4, 5)   => 12
  (1, 2)   => 1
  (6, 10)  => 17
  (8, 12)  => 19
  (12, 13) => 23
  (2, 5)   => 4
  (1, 3)   => 2
  (3, 7)   => 10
  (5, 9)   => 16
  (3, 8)   => 11
  (1, 4)   => 3
  (2, 6)   => 5
  (4, 6)   => 13
  (11, 13) => 22
  (9, 13)  => 20
  (4, 7)   => 14
  (2, 7)   => 6
  (4, 8)   => 15
  (2, 8)   => 7
  ⋮        => ⋮

### System Matrix 
Let's build a system matrix $\mathbf{A}$ for our graph model.

In [9]:
A = let

    # initialize -
    edges = directedgraphmodel.edges; # get the edges
    nodes = directedgraphmodel.nodes; # get the nodes
    edgesinverse = directedgraphmodel.edgesinverse; # get the inverse edges
    d = length(edges); # how many edges are there?
    n = length(nodes); # how many nodes are there?
    A = zeros(n, d+2); # initialize the system matrix, we have two extra columns for the input and output flows

    # fill the system matrix
    for (k,v) ∈ edgesinverse
        
        w = edges[v]; # weight of (u,v) edges
        A[v[1], k] = -w;
        A[v[2], k] = w;
    end

    # Update the columns for the input to node 1, and the exit from node 13 -
    A[1, d+1] = 1.0; # edge (∅,1)
    A[13, d+2] = -1.0; # edge (13,∅)

    A; # return
end

13×25 Matrix{Float64}:
 -1.0  -1.0  -1.0   0.0   0.0   0.0  …   0.0   0.0   0.0   0.0  1.0   0.0
  1.0   0.0   0.0  -1.0  -1.0  -1.0      0.0   0.0   0.0   0.0  0.0   0.0
  0.0   1.0   0.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0  0.0   0.0
  0.0   0.0   1.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0  0.0   0.0
  0.0   0.0   0.0   1.0   0.0   0.0      0.0   0.0   0.0   0.0  0.0   0.0
  0.0   0.0   0.0   0.0   1.0   0.0  …   0.0   0.0   0.0   0.0  0.0   0.0
  0.0   0.0   0.0   0.0   0.0   1.0      0.0   0.0   0.0   0.0  0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0  0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0     -1.0   0.0   0.0   0.0  0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0      0.0  -1.0   0.0   0.0  0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0  …   0.0   0.0  -1.0   0.0  0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0      0.0   0.0   0.0  -1.0  0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0      1.0   1.0   1.0   1.0  0.0  -1.0

Let's check a few nodes (rows) of the system matrix $\mathbf{A}$ to make sure it's constructed correctly. Consider two cases:

> __Test cases__
> 
> __Case 1:__ the source node `1` should have zero flows in, and an outgoing edge at nodes `2`, `3`, `4` where the coefficient $a_{ij}$ will be the negative edge weight $-w_{ij}$ (it is negative becaause it is leaving node 1).
>
> __Case 2:__  Any node not equal to `1` or `13` should have both incoming and outgoing edges, where the coefficients $a_{ij}$ will be the edge weights $w_{ij}$ for incoming edges and the negative edge weights $-w_{ij}$ for outgoing edges.

Let's check case 1:

In [10]:
let
    
    # initialize -
    test_node_index = 1; # we are looking at the source node 1
    test_row = A[test_node_index, :]; # row for node 1 (the source node);
    test_children_set = directedgraphmodel.children[test_node_index]; # children of node 1
    test_children_array = test_children_set |> collect |> sort; # convert to sorted array

    # test the edge to each of test_node_index children
    for i ∈ test_children_array
        edge_coordinates = (test_node_index, i);
        edge_weight = directedgraphmodel.edges[edge_coordinates];
        edge_index = edge_index_map[edge_coordinates];
        @assert test_row[edge_index] == -edge_weight "Error: edge weight mismatch at node $test_node_index to child $i"
    end

    # all tests passed
    println("All tests passed!")
end

All tests passed!


Next, let's check case 2. 

In [11]:
let
    
    # initialize -
    test_node_index = 2; # we are looking at a source node (any node not 1 or 13)
    test_row = A[test_node_index, :]; # row for node 1 (the source node);
    test_children_set = directedgraphmodel.children[test_node_index]; # children of node 1
    test_children_array = test_children_set |> collect |> sort; # convert to sorted array

    # find the nonzero elements of the test_row -
    test_nonzero_edge_indices = findall(!iszero, test_row);
    for i ∈ test_nonzero_edge_indices
        
        # get data for this edge -
        edge_coordinates = directedgraphmodel.edgesinverse[i]; # the (u,v) value for this edge index
        edge_weight = directedgraphmodel.edges[edge_coordinates]; # the weight of this edge

        

        # testing logic -
        if edge_coordinates[1] == test_node_index # this is an outgoing edge
            @assert test_row[i] == -edge_weight "Error: edge weight mismatch at node $test_node_index to child $(edge_coordinates[2])"
        elseif edge_coordinates[2] == test_node_index # this is an incoming edge
            @assert test_row[i] == edge_weight "Error: edge weight mismatch at node $test_node_index from parent $(edge_coordinates[1])"
        else
            error("Error: edge index $i does not correspond to node $test_node_index")
        end
    end
end

One more thing before we continue. The system matrix $\mathbf{A}$ is not square, as it has more columns than rows. This means we cannot use standard methods for solving linear systems directly. But can we do?

> __Degree of Freedom__ In this case, we have more flows than we have nodes. But all of methods require __square__ matrices. So we need to find a way to reduce the number of variables. In a practical siution like this, we could imagine that we __measure__ some of the flows, then these would no longer be unknows. We cpould use these measurments to reduce the number of variables in our system (so that we are left with a square matrix). 

But how many __degrees of freedom__, i.e., how many measurements do we need to make to fully determine the flow distribution? The naive answer would be to measure enough flows (columns) so that the number of unknown flows (columns) is equal to the number of nodes (rows).


In [17]:
let

    # initialize -
    (number_of_rows, number_of_columns) = size(A);

    # how many things do we need to measure?
    number_of_measurements = number_of_columns - number_of_rows;

    # let the user know how many measurements are needed
    println("Number of measurements needed: ", number_of_measurements);
end

Number of measurements needed: 12


Ok, great! But __which__ flows (columns) do we need to measure? 

> __Idea__: We want to measure flows that give a square matrix with maximum rank. This means we need to measure flows that are linearly independent from each other and from the existing flows in the system. That's where our QR pivoting method implemented in the `select_independent_columns(...)` function comes into play. This will compute a __non-uinique__ set of columns (if this set exists).

In the code block below, we compute the measured columns and then partition the original matrix `A` into two parts: `A₁` for the flows we want to compute and `A₂` for the flows we want to measure.

In [77]:
(A₁, A₂, keep, measure) = let

    # which columns should we measure?
    (number_of_rows, number_of_columns) = size(A);
    (keep, measure, r, tol) = select_independent_columns(A);

    if (r == min(number_of_rows, number_of_columns)) && (length(keep) == number_of_rows)
        println("We can measure the following flows (columns) to get a full-rank square matrix: ", measure);
    else
        @error "Matrix is rank deficient or we do not have enough independent columns"
    end

    # partition the system matrix
    A₁ = A[:, keep]; # corresponds to flows that we want to compute
    A₂ = A[:, measure]; # corresponds to flows that we want to measure

    (A₁, A₂, keep, measure) # return
end;

We can measure the following flows (columns) to get a full-rank square matrix: [4, 5, 6, 7, 9, 10, 11, 12, 14, 21, 23, 25]


### Hmmm. So where do we go now?
Now that we have partitioned our system in a measured matrix and a (square) system matrix. Thus, our original problem is now transformed:
$$
\begin{align*}
\mathbf{A}\mathbf{v} &= \mathbf{b}\quad\Longrightarrow\mathbf{A} = \mathbf{A}_{1}+\mathbf{A}_{2}\\
\mathbf{A}_{1}\mathbf{v}_{1} + \mathbf{A}_{2}\mathbf{v}_{2} &= \mathbf{b}\\
\mathbf{A}_{1}\mathbf{v}_{1} &= \mathbf{b} - \mathbf{A}_{2}\mathbf{v}_{2}
\end{align*}
$$
where $\mathbf{v}_{1}$ are the flows we want to compute, $\mathbf{v}_{2}$ are the flows that we measure and $\mathbf{b}$ is the right-hand side vector. Putting everyhting together gives the new system $\mathbf{A}_{1}\mathbf{v}_{1} = \mathbf{b}^{\prime}$:
$$
\boxed{
    \begin{align*}
    \mathbf{A}_{1}\mathbf{v}_{1} &= \underbrace{\mathbf{b} - \mathbf{A}_{2}\mathbf{v}_{2}}_{\mathbf{b}^{\prime}}
    \end{align*}\quad\blacksquare
}
$$
___

## Task 2: Setup the $\mathbf{b}^{\prime}$ vector and solve system using QR decomposition
In this task, we provide some dummy values for the measured flows $\mathbf{v}_{2}$ and use them to compute the new right-hand side vector $\mathbf{b}^{\prime}$. However, to make our life a little easier, be assume the original right-hand side vector $\mathbf{b} = \mathbf{0}$.

> __What does $\mathbf{b} = \mathbf{0}$ represent?__ Fill me in.

Let's setup our new right-hand side vector $\mathbf{b}^{\prime}$.

In [80]:
b′,v₂ = let

    number_of_measurements = length(measure);
    v₂ = rand(number_of_measurements); # dummy values for the measured flows
    b′ = -A₂ * v₂;
    b′,v₂ # return
end;

Next, given the new right-hand side vector $\mathbf{b}^{\prime}$, solve for $\mathbf{v}_{1}$ using QR decomposition. Let's see how this works.
> __QR Decomposition__
>
> The QR decomposition will factor a matrix into two orthogonal matrices, $\mathbf{Q}$ and $\mathbf{R}$, such that $\mathbf{A} = \mathbf{Q} \mathbf{R}$. 
> This is super handy for solving linear systems of equations:
>$$
>\begin{align*}
>\mathbf{A} \mathbf{x} &= \mathbf{b} \\
>\mathbf{Q} \mathbf{R} \mathbf{x} &= \mathbf{b} \\
>\mathbf{R} \mathbf{x} &= \mathbf{Q}^{\top} \mathbf{b} \\
>\mathbf{x} &= \mathbf{R}^{-1} \mathbf{Q}^{\top} \mathbf{b}\quad\blacksquare
>\end{align*}
>$$
>Thus, applying this idea to our system gives:
> $$
\boxed{
    \begin{align*}
    \mathbf{v}_1 &= \mathbf{R}^{-1} \mathbf{Q}^{\top} \left(\mathbf{b} - \mathbf{A}_2 \mathbf{v}_2\right)
    \end{align*}
> }
> $$

But, there is a wrinkle. We need to select a valid measurement vector $\mathbf{v}_2$ such that the system remains consistent. This means we need to ensure that the measured flows are compatible with the computed flows. In other words, we need to find a $\mathbf{v}_2$ that satisfies the original system of equations. Does such a $\mathbf{v}_2$ exist?

In [79]:
v₁ = let

    # do the decomposition -
    (Q,R) = qr(A₁);
    v₁ = R \ (transpose(Q) * (b′));

    v₁ # return
end

13-element Vector{Float64}:
  2.509678961987191
 -4.988592712949988
  0.07053592883521344
  1.4613068320040925
  0.9727233493853198
 -3.347338938327391
  0.931779777440229
 -3.6619369890211035
  1.4613068320040932
 -0.6464180767344887
 -3.3473389383273924
  1.1707290475361634
  0.018471020502251936

Fill me in

In [76]:
let
    residual = A₁ * v₁ + A₂ * v₂
end

13-element Vector{Float64}:
  1.3322676295501878e-15
 -4.440892098500626e-16
 -1.7763568394002505e-15
  8.881784197001252e-16
  0.0
  4.440892098500626e-16
  8.881784197001252e-16
  1.5543122344752192e-15
 -4.440892098500626e-16
 -2.220446049250313e-16
 -8.881784197001252e-16
  1.3322676295501878e-15
  4.440892098500626e-16

In [86]:
solution,measurements = let

    # initialize -
    should_stop_iterating = false;
    number_of_measurements = length(measure);
    (Q,R) = qr(A₁);
    counter=1;
    maximum_iterations = 1000; # set a maximum number of iterations to avoid infinite loops
    
    solution = nothing;
    measured_flows = nothing;
    while should_stop_iterating == false

        # do something here -
        v₂ = rand(number_of_measurements); # dummy values for the measured flows
        b′ = -A₂ * v₂;
        v₁ = R \ (transpose(Q) * (b′));
        
        if (any(v₁.<0) == false || counter >= maximum_iterations)
            # for now, we will just stop after one iteration
            should_stop_iterating = true; # set to true to stop iterating
            solution = v₁;
            measured_flows = v₂;
        else
            counter += 1; # increment the counter
        end
    end

    solution, measured_flows # return
end

([3.2794563346923256, -4.569637738816976, -0.7361503985942623, 2.2084169849852824, 0.32195711892687695, -3.398955756491282, 0.9547381598671801, -2.9673338106579914, 2.2084169849852833, -0.5683568819458173, -3.3989557564912825, -0.22596601674627648, 0.08615650728805924], [0.8922077126583349, 0.9502280852538006, 0.9560242821666035, 0.48099625461358686, 0.7406604732076425, 0.452325708692235, 0.40931774625910855, 0.2784742696673588, 0.8000669941264442, 0.9547381598671804, 0.32195711892687584, 0.08615650728805757])

In [87]:
solution

13-element Vector{Float64}:
  3.2794563346923256
 -4.569637738816976
 -0.7361503985942623
  2.2084169849852824
  0.32195711892687695
 -3.398955756491282
  0.9547381598671801
 -2.9673338106579914
  2.2084169849852833
 -0.5683568819458173
 -3.3989557564912825
 -0.22596601674627648
  0.08615650728805924

## Task 3: Solve the flow problem system using iteration
In this task, we'll solve the flow problem system using the iterative methods we developed in this module.

## Summary
Fill me in.